In [0]:
spark.conf.set(
    "fs.azure.account.key.retaildatalakeshawn123.dfs.core.windows.net",
    "SECRET"
)

In [0]:
#spit out couple lines of dataset
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://retail@retaildatalakeshawn123.dfs.core.windows.net/raw/online_retail.csv")

df.show(5)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|   17850.0|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|   17850.0|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
only showing top 5 rows


In [0]:
#create bronze layer
bronze_path = "abfss://retail@retaildatalakeshawn123.dfs.core.windows.net/bronze/online_retail_bronze"
df.write.format("delta") \
    .mode("overwrite") \
    .save(bronze_path)

bronze_df = spark.read.format("delta").load(bronze_path)
bronze_df.show(5)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|   17850.0|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|   17850.0|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|2010-12-01 08:26:00|     3.39|   17850.0|United Kingdom|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
only showing top 5 rows


In [0]:
#silver layer
from pyspark.sql.functions import col, to_timestamp, round

silver_df = bronze_df \
    .dropDuplicates() \
    .filter(col("CustomerID").isNotNull()) \
    .filter(col("Description").isNotNull()) \
    .filter(col("Quantity") > 0) \
    .filter(col("UnitPrice") > 0) \
    .withColumn("InvoiceDate", to_timestamp(col("InvoiceDate"))) \
    .withColumn("Revenue", round(col("Quantity") * col("UnitPrice"), 2))

silver_path = "abfss://retail@retaildatalakeshawn123.dfs.core.windows.net/silver/online_retail_silver"

silver_df.write.format("delta") \
    .mode("overwrite") \
    .save(silver_path)

silver_df = spark.read.format("delta").load(silver_path)

silver_df.show(5)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+-------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|Revenue|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+-------+
|   548327|    22429|ENAMEL MEASURING ...|       1|2011-03-30 13:05:00|     4.25|   16788.0|United Kingdom|   4.25|
|   548351|    21035|SET/2 RED RETROSP...|       6|2011-03-30 14:51:00|     3.25|   13001.0|United Kingdom|   19.5|
|   548466|   84997D|CHILDRENS CUTLERY...|       4|2011-03-31 12:19:00|     4.15|   16386.0|United Kingdom|   16.6|
|   548484|    22892|SET OF SALT AND P...|       2|2011-03-31 12:43:00|     1.25|   13571.0|United Kingdom|    2.5|
|   548501|    23077| DOUGHNUT LIP GLOSS |      20|2011-03-31 13:50:00|     1.25|   15894.0|United Kingdom|   25.0|
+---------+---------+--------------------+--------+-------------------+-

In [0]:
#gold layer for revenue
from pyspark.sql.functions import sum

gold_df = silver_df \
    .groupBy("CustomerID") \
    .agg(round(sum("Revenue"),2).alias("TotalRevenue")) \
    .orderBy(col("TotalRevenue").desc())

gold_path = "abfss://retail@retaildatalakeshawn123.dfs.core.windows.net/gold/top_customers"

gold_df.write.format("delta") \
    .mode("overwrite") \
    .save(gold_path)

gold_df.show(5)

+----------+------------+
|CustomerID|TotalRevenue|
+----------+------------+
|   14646.0|   280206.02|
|   18102.0|    259657.3|
|   17450.0|   194390.79|
|   16446.0|    168472.5|
|   14911.0|   143711.17|
+----------+------------+
only showing top 5 rows


In [0]:
#top products gold table
from pyspark.sql.functions import sum, col
top_products_df = silver_df \
    .groupBy("StockCode", "Description") \
    .agg(
        sum("Quantity").alias("TotalQuantitySold"),
        round(sum("Revenue"),2).alias("TotalRevenue")
    ) \
    .orderBy(col("TotalRevenue").desc())

top_products_df.show(5)

top_products_path = "abfss://retail@retaildatalakeshawn123.dfs.core.windows.net/gold/top_products"

top_products_df.write.format("delta") \
    .mode("overwrite") \
    .save(top_products_path)

+---------+--------------------+-----------------+------------+
|StockCode|         Description|TotalQuantitySold|TotalRevenue|
+---------+--------------------+-----------------+------------+
|    23843|PAPER CRAFT , LIT...|            80995|    168469.6|
|    22423|REGENCY CAKESTAND...|            12374|   142264.75|
|   85123A|WHITE HANGING HEA...|            36706|    100392.1|
|   85099B|JUMBO BAG RED RET...|            46078|    85040.54|
|    23166|MEDIUM CERAMIC TO...|            77916|    81416.73|
+---------+--------------------+-----------------+------------+
only showing top 5 rows


In [0]:
#monthly rev
from pyspark.sql.functions import year, month
monthly_revenue_df = silver_df \
    .withColumn("Year", year(col("InvoiceDate"))) \
    .withColumn("Month", month(col("InvoiceDate"))) \
    .groupBy("Year", "Month") \
    .agg(sum("Revenue").alias("MonthlyRevenue")) \
    .orderBy("Year", "Month")

monthly_revenue_df.show(5)


monthly_revenue_path = "abfss://retail@retaildatalakeshawn123.dfs.core.windows.net/gold/monthly_revenue"

monthly_revenue_df.write.format("delta") \
    .mode("overwrite") \
    .save(monthly_revenue_path)

+----+-----+-----------------+
|Year|Month|   MonthlyRevenue|
+----+-----+-----------------+
|2010|   12|570422.7299999952|
|2011|    1|568101.3099999962|
|2011|    2| 446084.919999996|
|2011|    3|594081.7599999963|
|2011|    4|468374.3299999959|
+----+-----+-----------------+
only showing top 5 rows
